# Esports Player Contract Prediction using Random Forest & GridSearchCV

## 1.Problem Statement

- The esports industry is growing rapidly, with organizations investing heavily in recruiting skilled players. Selecting the right player is a challenging task as it depends on multiple factors such as performance metrics, consistency, communication skills, and behavioral traits. Traditional recruitment methods rely on manual evaluation and subjective judgment, which may lead to biased or inefficient decisions.

- This project aims to develop a machine learning model that can predict whether an esports player should be offered a contract based on various performance and behavioral features, enabling data-driven and objective decision-making.

## 2.Objectives

- To build a classification model using Random Forest to predict whether a player should be signed (Contract_Signed).
- To preprocess and handle both numerical and categorical features using appropriate encoding techniques.
- To apply GridSearchCV for hyperparameter tuning in order to improve model performance.
- To evaluate the model using performance metrics such as accuracy, precision, recall, and F1-score.
- To analyze feature importance and identify the key factors influencing player selection decisions.

## 3.Import Libraries

In [142]:
import numpy as np
import pandas as pd

## 4.Load Dataset

In [143]:
df = pd.read_csv("esports_contract_dataset_clear.csv")
df.head(7)

,Reaction_Time,Accuracy,Matches_Played,Win_Rate,Avg_Damage,Team_Communication,Tilt_Rate,Tournament_Experience,Game_Type,Region,Playstyle,Contract_Signed
0,295,58,502,92.676783,696,6,0.197341,5,MOBA,Asia,Aggressive,1
1,316,56,1176,66.869539,697,5,0.122972,5,BattleRoyale,Asia,Aggressive,0
2,145,78,826,50.951491,677,9,0.615066,2,MOBA,Asia,Balanced,1
3,187,98,342,58.244878,866,8,0.518817,7,BattleRoyale,Asia,Aggressive,1
4,331,78,566,42.608103,792,9,0.530785,10,MOBA,NaN,Balanced,0
5,271,82,126,67.279515,647,5,0.538234,10,FPS,Europe,Balanced,0
6,223,67,111,66.833801,665,8,0.120675,6,FPS,Asia,Aggressive,1


In [144]:
df["Player_Efficiency"] = (
    df["Accuracy"] * df["Win_Rate"] / df["Reaction_Time"]
)

## 5.Check Missing Values & Handle

In [145]:
df.isnull().sum()

Reaction_Time              0
Accuracy                   0
Matches_Played             0
Win_Rate                   0
Avg_Damage                 0
Team_Communication         0
Tilt_Rate                  0
Tournament_Experience      0
Game_Type                  0
Region                   435
Playstyle                  0
Contract_Signed            0
Player_Efficiency          0
dtype: int64

In [146]:
df["Region"] = df['Region'].fillna(df['Region'].mode()[0], inplace=True)

C:\Users\yashk\AppData\Local\Temp\ipykernel_13996\3310281566.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df["Region"] = df['Region'].fillna(df['Region'].mode()[0], inplace=True)


In [147]:
df.isnull().sum()

Reaction_Time            0
Accuracy                 0
Matches_Played           0
Win_Rate                 0
Avg_Damage               0
Team_Communication       0
Tilt_Rate                0
Tournament_Experience    0
Game_Type                0
Region                   0
Playstyle                0
Contract_Signed          0
Player_Efficiency        0
dtype: int64

## 6.Preprocessing

In [148]:
df = pd.get_dummies(df,columns=["Game_Type","Region","Playstyle"]) # One-hot encoder
df

,Reaction_Time,Accuracy,Matches_Played,Win_Rate,Avg_Damage,Team_Communication,Tilt_Rate,Tournament_Experience,Contract_Signed,Player_Efficiency,Game_Type_BattleRoyale,Game_Type_FPS,Game_Type_MOBA,Region_Asia,Region_Europe,Playstyle_Aggressive,Playstyle_Balanced,Playstyle_Defensive
0,295,58,502,92.676783,696,6,0.197341,5,1,18.221198,False,False,True,True,False,True,False,False
1,316,56,1176,66.869539,697,5,0.122972,5,0,11.850298,True,False,False,True,False,True,False,False
2,145,78,826,50.951491,677,9,0.615066,2,1,27.408388,False,False,True,True,False,False,True,False
3,187,98,342,58.244878,866,8,0.518817,7,1,30.524054,True,False,False,True,False,True,False,False
4,331,78,566,42.608103,792,9,0.530785,10,0,10.040580,False,False,True,False,True,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1495,241,69,990,80.927952,347,2,0.630170,6,0,23.170244,False,False,True,False,True,True,False,False
1496,158,68,118,30.175329,1180,1,0.269487,0,0,12.986851,False,True,False,False,True,True,False,False
1497,120,65,322,58.405541,835,5,0.835201,8,1,31.636335,True,False,False,False,True,False,False,True
1498,340,60,137,75.270024,697,9,0.355884,8,1,13.282945,False,True,False,False,True,False,False,True


## 7.Model Building

### i) Seperate Target & Feature

In [149]:
X = df.drop("Contract_Signed", axis=1)
y = df["Contract_Signed"]

### ii) Split The Dataset

In [150]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.20,random_state=42)

### iii) Select Model

In [151]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()

### iv) GridSearchCV

In [152]:
from sklearn.model_selection import GridSearchCV
param_grid = {
    'n_estimators': [200, 300],
    'max_depth': [10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt'],
    'bootstrap': [True, False]
}

grid = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

### v) Train Model

In [153]:
grid.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=RandomForestClassifier(), n_jobs=-1,
             param_grid={'bootstrap': [True, False], 'max_depth': [10, 15, 20],
                         'max_features': ['sqrt'],
                         'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [200, 300]},
             scoring='f1')

### vi) Get Best Model

In [154]:
print(grid.best_params_)
print(grid.best_score_)

{'bootstrap': True, 'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
0.8606388340185156


## 8.Predict

In [155]:
y_pred = grid.predict(X_test)





## 9.Evaluate

In [156]:
from sklearn.metrics import accuracy_score,recall_score,f1_score,precision_score

A = accuracy_score(y_test,y_pred)
B = precision_score(y_test,y_pred)
C = f1_score(y_test,y_pred)
D = recall_score(y_test,y_pred)


print("Accuracy:",A)
print("Precision:",B)
print("Recall:",D)
print("F1:",C)

Accuracy: 0.8666666666666667
Precision: 0.8571428571428571
Recall: 0.8571428571428571
F1: 0.8571428571428571


## 10.Conclusion:

This project successfully developed a machine learning model to predict whether an esports player should be offered a contract based on performance and behavioral attributes. A Random Forest Classifier was implemented along with GridSearchCV for hyperparameter tuning to achieve optimal performance.

Initially, the model showed moderate performance due to weak feature-target relationships and high data noise. By refining the dataset and ensuring clearer patterns, the model performance improved significantly. The final model achieved an accuracy of 86% along with a balanced precision, recall, and F1-score of 0.85, indicating strong and reliable predictions.

The project demonstrates the importance of data quality, feature engineering, and proper model tuning in building effective machine learning solutions. It also highlights how machine learning can assist esports organizations in making data-driven and objective recruitment decisions.

Overall, this project provides a complete end-to-end pipeline, including data preprocessing, feature engineering, model training, hyperparameter tuning, and evaluation, making it a practical and impactful application of machine learning.